In [1]:
import pandas as pd

# Load training dataset
train_df = pd.read_csv("data/train.csv")

train_df.head()


,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week,income,workclass_Local-gov,workclass_Never-worked,workclass_Private,...,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia,native.country_nan
0,62,109463,10,0,1617,33,0,False,False,True,...,False,False,False,False,False,False,True,False,False,False
1,52,312477,9,0,0,40,0,False,False,True,...,False,False,False,False,False,False,True,False,False,False
2,69,168794,4,0,0,48,0,False,False,False,...,False,False,False,False,False,False,True,False,False,False
3,33,199227,13,0,0,55,1,False,False,True,...,False,False,False,False,False,False,True,False,False,False
4,40,374367,11,0,0,44,0,False,False,True,...,False,False,False,False,False,False,True,False,False,False


In [2]:
train_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22775 entries, 0 to 22774
Columns: 101 entries, age to native.country_nan
dtypes: bool(94), int64(7)
memory usage: 3.3 MB


In [3]:

train_df.columns.tolist()


['age',
 'fnlwgt',
 'education.num',
 'capital.gain',
 'capital.loss',
 'hours.per.week',
 'income',
 'workclass_Local-gov',
 'workclass_Never-worked',
 'workclass_Private',
 'workclass_Self-emp-inc',
 'workclass_Self-emp-not-inc',
 'workclass_State-gov',
 'workclass_Without-pay',
 'workclass_nan',
 'education_11th',
 'education_12th',
 'education_1st-4th',
 'education_5th-6th',
 'education_7th-8th',
 'education_9th',
 'education_Assoc-acdm',
 'education_Assoc-voc',
 'education_Bachelors',
 'education_Doctorate',
 'education_HS-grad',
 'education_Masters',
 'education_Preschool',
 'education_Prof-school',
 'education_Some-college',
 'marital.status_Married-AF-spouse',
 'marital.status_Married-civ-spouse',
 'marital.status_Married-spouse-absent',
 'marital.status_Never-married',
 'marital.status_Separated',
 'marital.status_Widowed',
 'occupation_Armed-Forces',
 'occupation_Craft-repair',
 'occupation_Exec-managerial',
 'occupation_Farming-fishing',
 'occupation_Handlers-cleaners',
 'oc

In [4]:

X = train_df.drop("income", axis=1)
y = train_df["income"]

X.shape, y.shape


((22775, 100), (22775,))

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Train-test split
X_train, X_val, Y_train, Y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, Y_train)

print("Validation Accuracy:", model.score(X_val, Y_val))


Validation Accuracy: 0.8006586169045006


In [7]:
import os
import joblib

os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/income_model.pkl")

print("Model Saved!")


Model Saved!


In [8]:
sample = X.iloc[[0]]
model.predict(sample)


array([1], dtype=int64)

In [10]:
import pandas as pd
import joblib
import time

# Load model
model = joblib.load("models/income_model.pkl")

# Load stream data
stream_df = pd.read_csv("data/stream.csv")


# ---- Sensitive Attribute Decoders ----

def decode_gender(row):
    # Your dataset only has sex_Male
    return "Male" if row["sex_Male"] == 1 else "Female"

def decode_race(row):
    race_cols = ["race_Asian-Pac-Islander", "race_Black", "race_Other", "race_White"]
    for col in race_cols:
        if row[col] == 1:
            return col.replace("race_", "")
    return "Unknown"


# ---- Streaming Simulation ----

print("Starting streaming simulation...\n")

for idx, row in stream_df.head(10).iterrows():  # first 10 rows to test
    x = row.drop("income", errors="ignore")  # features only
    
    pred = int(model.predict(pd.DataFrame([x]))[0])

    
    gender = decode_gender(row)
    race = decode_race(row)

    print(f"{idx}: PRED = {pred} | {gender} | {race}")

    time.sleep(0.5)  # simulate stream (later will be 1 sec)


Starting streaming simulation...

0: PRED = 0 | Male | White
1: PRED = 0 | Male | White
2: PRED = 0 | Female | White
3: PRED = 0 | Male | Black
4: PRED = 0 | Female | White
5: PRED = 0 | Female | White
6: PRED = 0 | Male | White
7: PRED = 0 | Female | White
8: PRED = 0 | Female | Black
9: PRED = 0 | Female | White
